In [1]:
!pip install -q transformers
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q pydantic
!pip install -q pandas
!pip install -q accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 39.1 MB/s eta 0:00:00


In [2]:
import json
import pandas as pd
import numpy as np

from transformers import pipeline

from pydantic import BaseModel, ValidationError
from typing import Literal

from sentence_transformers import SentenceTransformer

import faiss

In [3]:
import json
import pandas as pd
import numpy as np

from transformers import pipeline

from pydantic import BaseModel, ValidationError
from typing import Literal

from sentence_transformers import SentenceTransformer

import faiss

In [4]:
print("All libraries imported successfully.")

All libraries imported successfully.


In [5]:
import pandas as pd

patient_notes = [
    {"id": 1, "patient_note": "A 58-year-old male reports severe chest pain during exercise with shortness of breath."},
    {"id": 2, "patient_note": "The patient has high blood pressure and elevated cholesterol with occasional dizziness."},
    {"id": 3, "patient_note": "A 45-year-old female experiences mild chest discomfort while climbing stairs."},
    {"id": 4, "patient_note": "The patient complains of fatigue and irregular heartbeat for the last two weeks."},
    {"id": 5, "patient_note": "A 63-year-old male has diabetes, chest pain, and abnormal ECG findings."},
    {"id": 6, "patient_note": "The patient reports no chest pain but has persistent high blood pressure."},
    {"id": 7, "patient_note": "A 54-year-old female experiences shortness of breath during daily activities."},
    {"id": 8, "patient_note": "The patient has normal ECG results and no significant cardiac symptoms."},
    {"id": 9, "patient_note": "A 60-year-old male reports chest pain radiating to the left arm with sweating."},
    {"id": 10, "patient_note": "The patient feels occasional dizziness and mild fatigue after walking."},
    {"id": 11, "patient_note": "A 49-year-old female has palpitations and shortness of breath during exercise."},
    {"id": 12, "patient_note": "The patient has elevated cholesterol but no current symptoms of heart disease."},
    {"id": 13, "patient_note": "A 67-year-old male experiences severe chest pain even while resting."},
    {"id": 14, "patient_note": "The patient reports mild chest discomfort with normal blood pressure."},
    {"id": 15, "patient_note": "A 56-year-old female has fatigue, dizziness, and abnormal stress test results."}
]

df = pd.DataFrame(patient_notes)

df.head()

,id,patient_note
0,1,A 58-year-old male reports severe chest pain d...
1,2,The patient has high blood pressure and elevat...
2,3,A 45-year-old female experiences mild chest di...
3,4,The patient complains of fatigue and irregular...
4,5,"A 63-year-old male has diabetes, chest pain, a..."


In [7]:
print(df.shape)


(15, 2)


In [8]:
df.to_csv("patient_notes.csv", index=False)

print("Dataset saved successfully.")

Dataset saved successfully.


In [9]:
df = pd.read_csv("patient_notes.csv")

df.head()

,id,patient_note
0,1,A 58-year-old male reports severe chest pain d...
1,2,The patient has high blood pressure and elevat...
2,3,A 45-year-old female experiences mild chest di...
3,4,The patient complains of fatigue and irregular...
4,5,"A 63-year-old male has diabetes, chest pain, a..."


# Step 3: Load LLM

In [12]:
!pip install -q torch transformers accelerate sentencepiece

In [13]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

In [14]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

generator = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer
)

print("Model Loaded Successfully")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model Loaded Successfully


In [16]:
note = df.loc[0, "patient_note"]

prompt = f"""
You are a cardiologist.

Extract the information from the patient note.

Return ONLY valid JSON.

Patient Note:
{note}

JSON format:

{{
  "risk_level": "",
  "primary_symptom": "",
  "urgency": "",
  "summary": ""
}}
"""

result = generator(
    prompt,
    max_new_tokens=150,
    do_sample=False,
    return_full_text=False
)

print(result[0]["generated_text"])

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


```json
{
  "risk_level": "High",
  "primary_symptom": "Severe chest pain during exercise and shortness of breath",
  "urgency": "Immediate",
  "summary": "A 58-year-old male presents with severe chest pain during exercise accompanied by shortness of breath. This is an urgent condition requiring immediate medical attention."
}
```


## Step 4: Define Pydantic Schema for Structured Validation

In [17]:
from pydantic import BaseModel
from typing import Literal

class PatientInfo(BaseModel):

    risk_level: Literal["Low", "Medium", "High"]

    primary_symptom: str

    urgency: Literal["Low", "Medium", "High"]

    summary: str

print("Schema Created Successfully")

Schema Created Successfully


In [21]:
import json

response = result[0]["generated_text"]

# Remove markdown code block
response = response.replace("```json", "").replace("```", "").strip()

# Convert JSON string to Python dictionary
data = json.loads(response)

print(data)

{'risk_level': 'High', 'primary_symptom': 'Severe chest pain during exercise and shortness of breath', 'urgency': 'Immediate', 'summary': 'A 58-year-old male presents with severe chest pain during exercise accompanied by shortness of breath. This is an urgent condition requiring immediate medical attention.'}


## Create Pydantic Schema

In [28]:
from pydantic import BaseModel, field_validator
from typing import Literal

class HeartDiseaseExtraction(BaseModel):
    risk_level: str
    primary_symptom: str
    urgency: str
    summary: str

    @field_validator('urgency')
    @classmethod
    def validate_urgency(cls, v: str) -> str:
        mapping = {
            "Immediate": "High",
            "Critical": "High",
            "Urgent": "High",
            "Moderate": "Medium",
            "Minor": "Low"
        }
        normalized = mapping.get(v, v)
        if normalized not in ["Low", "Medium", "High"]:
            # Default to Medium if unknown but non-empty
            return "Medium"
        return normalized

    @field_validator('risk_level')
    @classmethod
    def validate_risk(cls, v: str) -> str:
        if v not in ["Low", "Medium", "High"]:
             return "Medium"
        return v

In [29]:
from pydantic import ValidationError

try:
    validated_data = HeartDiseaseExtraction(**data)

    print("Validation Successful")
    print(validated_data)

except ValidationError as e:

    print("Validation Failed")
    print(e)

    # Normalize invalid urgency values
    urgency_mapping = {
        "Immediate": "High",
        "Critical": "High",
        "Urgent": "High",
        "Moderate": "Medium",
        "Minor": "Low"
    }

    if data["urgency"] in urgency_mapping:
        data["urgency"] = urgency_mapping[data["urgency"]]

    print("\nAfter Normalization\n")

    validated_data = HeartDiseaseExtraction(**data)

    print("Validation Successful")
    print(validated_data)

Validation Successful
risk_level='High' primary_symptom='Severe chest pain during exercise and shortness of breath' urgency='High' summary='A 58-year-old male presents with severe chest pain during exercise accompanied by shortness of breath. This is an urgent condition requiring immediate medical attention.'


## Function

In [30]:
import json
from pydantic import ValidationError

def process_note(note):
    prompt = f"""
    You are an experienced cardiologist.
    Read the patient note carefully.
    Return ONLY valid JSON.

    Patient Note:
    {note}

    Output format:
    {{
      "risk_level": "Low/Medium/High",
      "primary_symptom": "",
      "urgency": "Low/Medium/High",
      "summary": ""
    }}
    """

    result = generator(
        prompt,
        max_new_tokens=150,
        do_sample=False,
        return_full_text=False
    )

    response = result[0]["generated_text"]

    # Improved markdown cleaning
    if "```json" in response:
        response = response.split("```json")[1].split("```")[0]
    elif "```" in response:
        response = response.split("```")[1].split("```")[0]

    response = response.strip()

    try:
        data = json.loads(response)
        # Validation and normalization happen inside the Pydantic model now
        validated = HeartDiseaseExtraction(**data)
        return validated.model_dump()
    except Exception as e:
        print(f"Parsing/Validation error: {e}")
        raise e

In [31]:
results = []

for index, row in df.iterrows():

    print(f"Processing Record {row['id']}")

    try:

        output = process_note(row["patient_note"])

        output["id"] = int(row["id"])

        results.append(output)

    except Exception as e:

        print(f"Error in Record {row['id']}")

        print(e)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Processing Record 1


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Processing Record 2


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Processing Record 13


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Processing Record 14


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Processing Record 15


In [32]:
results

[{'risk_level': 'Medium',
  'primary_symptom': 'Severe chest pain and shortness of breath during exercise',
  'urgency': 'High',
  'summary': 'The patient presents with severe symptoms indicating a potential cardiac issue. Urgent medical attention is required.',
  'id': 1},
 {'risk_level': 'Medium',
  'primary_symptom': 'Dizziness',
  'urgency': 'Medium',
  'summary': 'The patient has high blood pressure and elevated cholesterol, which may lead to cardiovascular issues. They also experience occasional dizziness.',
  'id': 2},
 {'risk_level': 'Medium',
  'primary_symptom': 'Mild chest discomfort during activity',
  'urgency': 'Medium',
  'summary': 'The patient presents with mild chest discomfort, which is a common symptom of angina. The risk level is considered medium due to the presence of symptoms during physical exertion. Urgent medical attention may be required if the symptoms worsen or persist.',
  'id': 3},
 {'risk_level': 'Medium',
  'primary_symptom': 'Fatigue and Irregular Hea

##

In [34]:
bad_record = {
    "risk_level": "Critical",
    "primary_symptom": "Chest Pain",
    "urgency": "Immediate",
    "summary": "Patient has severe chest pain."
}

In [36]:
import json

In [37]:
with open("results.json", "w") as f:
    json.dump(results, f, indent=4)

print("results.json saved successfully.")

results.json saved successfully.


In [38]:
with open("results.json", "r") as f:
    data = json.load(f)

print(data[:2])      # First 2 records

[{'risk_level': 'Medium', 'primary_symptom': 'Severe chest pain and shortness of breath during exercise', 'urgency': 'High', 'summary': 'The patient presents with severe symptoms indicating a potential cardiac issue. Urgent medical attention is required.', 'id': 1}, {'risk_level': 'Medium', 'primary_symptom': 'Dizziness', 'urgency': 'Medium', 'summary': 'The patient has high blood pressure and elevated cholesterol, which may lead to cardiovascular issues. They also experience occasional dizziness.', 'id': 2}]


In [39]:
import json

with open("results.json", "w") as f:
    json.dump(results, f, indent=4)

print("results.json saved successfully.")

results.json saved successfully.


In [40]:
with open("results.json", "r") as f:
    data = json.load(f)

print(data[0])

{'risk_level': 'Medium', 'primary_symptom': 'Severe chest pain and shortness of breath during exercise', 'urgency': 'High', 'summary': 'The patient presents with severe symptoms indicating a potential cardiac issue. Urgent medical attention is required.', 'id': 1}


In [41]:
print("Malformed Record")
print(bad_record)

try:
    validated = HeartDiseaseExtraction(**bad_record)

    print("\nValidation Successful")
    print(validated)

except Exception as e:

    print("\nValidation Failed")
    print(e)

Malformed Record
{'risk_level': 'Critical', 'primary_symptom': 'Chest Pain', 'urgency': 'Immediate', 'summary': 'Patient has severe chest pain.'}

Validation Successful
risk_level='Medium' primary_symptom='Chest Pain' urgency='High' summary='Patient has severe chest pain.'
